# 00 — Google ADK setup check

## Goal

Pass a credential-free setup check first, then optionally run one small ADK invocation with your own local Gemini API key. This notebook is intentionally a teaching aid, not an agent starter project.

It shows a redacted runtime timeline: user input, agent, tool call, tool result, and a final-answer acknowledgement. It never prints API keys, raw events, private reasoning, hidden instructions, arbitrary tool data, or live model text.

## 1. Setup

Run this notebook from the Search Agent Lab repository with its local virtual environment selected as the kernel. Offline mode is the default and does not contact a model or require a key. Set GITHUB_USERNAME explicitly to your public GitHub login before the optional live run. Set RUN_LIVE_CHECK to True only after adding GOOGLE_API_KEY to the untracked repository-root .env file.

In [ ]:
from collections.abc import Mapping
from importlib.metadata import version
from pathlib import Path
import os
import re
import sys

from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.apps import App
from google.adk.events import Event
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.env.example').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise RuntimeError('Run this notebook from inside the Search Agent Lab repository.')


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from search_agent_lab.week1_checkpoint import (
    ACHIEVEMENT_MESSAGE,
    build_issue_form_url,
    generate_codename,
    normalize_github_username,
)

load_dotenv(REPOSITORY_ROOT / '.env', override=False)

RUN_LIVE_CHECK = False
GITHUB_USERNAME = ''  # Set this explicitly, for example: 'octocat'
LIVE_MODEL = 'gemini-flash-latest'
AGENT_NAME = 'setup_agent'
PUBLIC_TOPIC = 'google-adk'

credential_available = bool(os.environ.get('GOOGLE_API_KEY'))
credential_status = 'detected locally' if credential_available else 'not configured'
print(f'google-adk {version("google-adk")} | credential: {credential_status}')
print(f'live check enabled: {RUN_LIVE_CHECK}')

## 2. Deterministic local tool

The offline check calls this typed function directly. Its input and output are deliberately allowlisted public values, so the notebook can safely demonstrate the tool boundary without a network request.

In [ ]:
def lookup_lab_status(topic: str) -> dict[str, str]:
    '''Return the fixed public status used by this setup check.'''
    if topic != PUBLIC_TOPIC:
        return {
            'status': 'unsupported',
            'topic': 'redacted',
            'summary': 'Only the public lab topic is supported.',
        }
    return {
        'status': 'ready',
        'topic': PUBLIC_TOPIC,
        'summary': 'The deterministic local tool completed.',
    }

## 3. Redacted event view

ADK events contain more than a learner needs to see. These helpers inspect only a known tool name, one allowlisted argument, three allowlisted result fields, and the presence of non-thought final text. Unknown tools and event errors remain redacted.

In [ ]:
GOOGLE_API_KEY_PATTERN = re.compile(r'AIza[0-9A-Za-z_-]{20,}')
NAMED_SECRET_PATTERN = re.compile(
    r'(?i)\b(api[ _-]?key|token|secret|password|authorization)\s*[:=]\s*[^\s,;]+'
)


def compact_public_text(value: object, limit: int = 240) -> str:
    if not isinstance(value, str):
        return 'redacted'
    text = ' '.join(value.split())
    text = GOOGLE_API_KEY_PATTERN.sub('<redacted-google-api-key>', text)
    text = NAMED_SECRET_PATTERN.sub(r'\1=<redacted>', text)
    return text[:limit]


def public_topic(value: object) -> str:
    return PUBLIC_TOPIC if value == PUBLIC_TOPIC else 'redacted'


def redact_tool_call(call: object) -> dict[str, object]:
    if getattr(call, 'name', None) != 'lookup_lab_status':
        return {'redacted': True}
    arguments = getattr(call, 'args', None)
    topic = arguments.get('topic') if isinstance(arguments, Mapping) else None
    return {'tool': 'lookup_lab_status', 'arguments': {'topic': public_topic(topic)}}


def redact_tool_result(response: object) -> dict[str, object]:
    if getattr(response, 'name', None) != 'lookup_lab_status':
        return {'redacted': True}
    raw_result = getattr(response, 'response', None)
    if not isinstance(raw_result, Mapping):
        return {'tool': 'lookup_lab_status', 'result': {'status': 'redacted'}}
    return {
        'tool': 'lookup_lab_status',
        'result': {
            'status': compact_public_text(raw_result.get('status')),
            'topic': public_topic(raw_result.get('topic')),
            'summary': compact_public_text(raw_result.get('summary')),
        },
    }


def has_nonthought_final_text(event: Event) -> bool:
    content = getattr(event, 'content', None)
    parts = getattr(content, 'parts', None) or []
    return any(
        getattr(part, 'text', None) and not getattr(part, 'thought', False)
        for part in parts
    )


def redacted_event_rows(event: Event) -> list[tuple[str, object]]:
    rows: list[tuple[str, object]] = []
    for call in event.get_function_calls():
        rows.append(('tool call', redact_tool_call(call)))
    for response in event.get_function_responses():
        rows.append(('tool result', redact_tool_result(response)))
    if getattr(event, 'error_code', None):
        rows.append(('runtime error', 'redacted ADK error'))
    if event.author != 'user' and event.is_final_response() and has_nonthought_final_text(event):
        rows.append(('final answer', 'Final response received (content omitted by setup check).'))
    return rows


def render_timeline(rows: list[tuple[str, object]]) -> None:
    print('Redacted runtime timeline')
    for stage, value in rows:
        print(f'{stage:13} | {value}')

## 4. Offline/mock check

This cell is the baseline test. It invokes the local tool directly, builds safe synthetic ADK event shapes, and verifies that the redacted timeline has every teaching step. No Gemini client is constructed and no network request is made.

In [ ]:
OFFLINE_INPUT = 'Run the credential-free Google ADK setup check.'
offline_tool_result = lookup_lab_status(PUBLIC_TOPIC)

offline_events = [
    Event(
        author=AGENT_NAME,
        content=types.Content(
            parts=[
                types.Part(
                    function_call=types.FunctionCall(
                        name='lookup_lab_status',
                        args={'topic': PUBLIC_TOPIC},
                    )
                )
            ]
        ),
    ),
    Event(
        author=AGENT_NAME,
        content=types.Content(
            parts=[
                types.Part(
                    function_response=types.FunctionResponse(
                        name='lookup_lab_status',
                        response=offline_tool_result,
                    )
                )
            ]
        ),
    ),
    Event(
        author=AGENT_NAME,
        content=types.Content(parts=[types.Part(text='The local setup check is ready.')]),
    ),
]

offline_timeline: list[tuple[str, object]] = [
    ('user input', OFFLINE_INPUT),
    ('agent', AGENT_NAME),
]
for offline_event in offline_events:
    offline_timeline.extend(redacted_event_rows(offline_event))

render_timeline(offline_timeline)
expected_stages = ['user input', 'agent', 'tool call', 'tool result', 'final answer']
assert [stage for stage, _ in offline_timeline] == expected_stages
assert offline_tool_result['status'] == 'ready'
print('Offline setup check: PASS')

## 5. Optional live ADK check

Leave RUN_LIVE_CHECK set to False for the normal onboarding run; that path never creates a codename or submission link. To opt in, set GITHUB_USERNAME to your public GitHub login, then set RUN_LIVE_CHECK to True after creating the untracked .env file. The checkpoint appears only after the credential is detected, the live invocation completes, and real tool-call, tool-result, and final-answer events are present. Submission is a public, optional honor-system celebration—not authentication or formal grading.

In [ ]:
LIVE_INPUT = 'Use lookup_lab_status for the public topic google-adk, then give one short setup status.'


async def run_live_setup_check() -> list[tuple[str, object]]:
    live_agent = LlmAgent(
        name=AGENT_NAME,
        model=LIVE_MODEL,
        instruction=(
            'You are a setup checker. Call lookup_lab_status exactly once with '
            'topic google-adk, then return one short public sentence.'
        ),
        tools=[lookup_lab_status],
        generate_content_config=types.GenerateContentConfig(temperature=0),
    )
    app = App(name='setup_check', root_agent=live_agent)
    session_service = InMemorySessionService()
    session_id = 'setup-check'
    await session_service.create_session(
        app_name=app.name,
        user_id='learner',
        session_id=session_id,
    )
    runner = Runner(app=app, session_service=session_service)
    timeline: list[tuple[str, object]] = [
        ('user input', LIVE_INPUT),
        ('agent', AGENT_NAME),
    ]
    async for event in runner.run_async(
        user_id='learner',
        session_id=session_id,
        new_message=types.Content(role='user', parts=[types.Part(text=LIVE_INPUT)]),
    ):
        timeline.extend(redacted_event_rows(event))
    return timeline


if not RUN_LIVE_CHECK:
    print('Live check skipped. Set RUN_LIVE_CHECK = True only after local credential setup.')
elif not credential_available:
    print('Live check skipped. GOOGLE_API_KEY is not configured locally.')
else:
    try:
        live_timeline = await run_live_setup_check()
    except Exception:
        print('Live check did not complete. Run offline mode, then verify local key, network, quota, and model access.')
    else:
        render_timeline(live_timeline)
        required_stages = {'tool call', 'tool result', 'final answer'}
        observed_stages = {stage for stage, _ in live_timeline}
        if not required_stages.issubset(observed_stages):
            print('Checkpoint not generated: the complete tool-use timeline was not observed.')
        else:
            print(ACHIEVEMENT_MESSAGE)
            if not GITHUB_USERNAME.strip():
                print('Set GITHUB_USERNAME explicitly near the top of the notebook, then rerun this cell.')
            else:
                try:
                    normalized_username = normalize_github_username(GITHUB_USERNAME)
                except ValueError:
                    print('GITHUB_USERNAME is invalid. Use only your public GitHub login.')
                else:
                    checkpoint_codename = generate_codename(normalized_username)
                    submission_url = build_issue_form_url(normalized_username)
                    print(f'Week 1 agent codename: {checkpoint_codename}')
                    print(f'Optional public Issue Form: {submission_url}')
                    print('Public honor-system celebration only; not authentication or formal grading.')

## Next steps

- Offline PASS means the local Python, ADK package, deterministic tool, and safe timeline path are ready.
- A successful optional live check confirms only the locally configured Gemini path; it does not authorize deployment or Google Cloud setup.
- Keep .env untracked. Do not save or commit notebook outputs if they contain unexpected external content.
- The upstream shop_agent.ipynb will be studied later from a pinned separate checkout, not copied into this repository.